# ABNB hotels — Supra email sample

Export a sample of Airbnb emails from NIQ's Supra subject-line feed (`SHARE_NIQ.SRS.SUPRA_QUALITY_MERCHANT_*`) to see what each template carries and where a hotel stay shows up.

- **Week pull:** every Airbnb email sent in one week, joined to its subject header (subject, sender, hashed-token info).
- **Template summary:** subjects collapsed to templates (digits → `#`), with volume, mailboxes, senders and hotel-keyword hits.
- **Stratified sample:** up to `SAMPLE_PER_TEMPLATE` emails per template, plus every hotel-keyword email.
- **Mailbox traces:** the full Airbnb email history of a few mailboxes that got a hotel-like review request that week, and of a matched set that got a non-hotel one, to follow a stay from booking to review.

Notes from 2026-10-14: blanked words (`__`) keep a consistent hash in the header `aux_data` (`subject_hash_info.hashed_tokens` + `token_lengths`), but those hashes do not match receipt `reservation_number`s. The emails ⨝ header join times out on the MCP for more than ~1 week, so pulls stay week-sized.

Outputs go to `hotels/.cache/supra_sample/` (gitignored).

In [1]:
import json
from pathlib import Path

import pandas as pd
import snowflake.connector

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 100)

# Keyring CredWrite fails on this machine (WinError 1783), so don't cache the token.
conn = snowflake.connector.connect(connection_name="default", client_store_temporary_credential=False)


def q(sql: str) -> pd.DataFrame:
    df = conn.cursor().execute(sql).fetch_pandas_all()
    df.columns = [c.lower() for c in df.columns]
    return df


OUT = Path("hotels/.cache/supra_sample") if Path("hotels").is_dir() else Path(".cache/supra_sample")
OUT.mkdir(parents=True, exist_ok=True)
OUT.resolve()

WindowsPath('C:/Users/fintan.creedon/code/exploration/abnb/hotels/.cache/supra_sample')

In [2]:
WEEK_START, WEEK_END = "2026-09-01", "2026-09-07"
SAMPLE_PER_TEMPLATE = 20
N_TRACE_MAILBOXES = 25
TRACE_LOOKBACK_DAYS = 120
SEED = 7

HEADER = "SHARE_NIQ.SRS.SUPRA_QUALITY_MERCHANT_EMAIL_HEADER"
EMAILS = "SHARE_NIQ.SRS.SUPRA_QUALITY_MERCHANT_EMAILS"

# Host-name keywords used for the hotel share on 2026-10-14.
HOTEL_RE = r"\b(hotel|hotels|inn|motel|suites|resort|hostel)\b"

## 1. One week of Airbnb emails

In [3]:
COLS = """
    e.email_id, e.email_sent_date, e.user_id, e.mailbox_id, e.email_type, e.email_type_confidence,
    e.status AS email_status, e.aux_data AS email_aux, e.subject_id,
    h.subject, h.from_addr, h.aux_data AS header_aux
"""

# Filter emails on date first: an unfiltered join to the header times out.
week = q(f"""
    WITH e AS (SELECT * FROM {EMAILS} WHERE email_sent_date BETWEEN '{WEEK_START}' AND '{WEEK_END}'),
         h AS (SELECT subject_id, subject, from_addr, aux_data FROM {HEADER} WHERE domain = 'airbnb')
    SELECT {COLS}
    FROM e JOIN h ON e.subject_id = h.subject_id
""")


def enrich(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["template"] = df["subject"].str.replace(r"\d+", "#", regex=True)
    df["hotel_kw"] = df["subject"].str.contains(HOTEL_RE, case=False, regex=True)
    df["n_blanks"] = df["subject"].str.count("__")
    return df


week = enrich(week)
print(f"{len(week):,} emails, {week.mailbox_id.nunique():,} mailboxes, {week.template.nunique():,} templates")
week.head()

C:\Users\fintan.creedon\AppData\Local\Temp\ipykernel_42464\806143361.py:19: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["hotel_kw"] = df["subject"].str.contains(HOTEL_RE, case=False, regex=True)


280,640 emails, 108,331 mailboxes, 28,921 templates


,email_id,email_sent_date,user_id,mailbox_id,email_type,email_type_confidence,email_status,email_aux,subject_id,subject,from_addr,header_aux,template,hotel_kw,n_blanks
0,abac9b55-c47a-34eb-b99b-7827499974ea,2026-09-06,0083b143-099e-3bbf-b243-1d98119c8397,cfd5e00b-e93d-3c20-a6d1-83647e304bbc,NOT_A_RECEIPT,1.000007,I,None,90ce3bd0-90ef-53fc-bd52-696a6c5d0802,__ de reserva __ para __ Nido __ A un Paso del __ y __,automated@airbnb.com,"{\n ""subject_hash_info"": {\n ""hashed_tokens"": [\n ""hNK9KBQIkqk1kF40EcUD"",\n ""88EfH...",__ de reserva __ para __ Nido __ A un Paso del __ y __,False,6
1,1acc24cc-b188-3a2b-89cc-b321da47a9fc,2026-09-02,f19cd393-5120-3032-a28d-babd5358e1e3,b42a437b-5b0c-3d41-aa1a-9dfc0c2a28d9,NOT_A_RECEIPT,1.000010,I,None,47b67ba9-f50e-5e03-a534-ca2a6047834c,__ Island homes you should see,discover@airbnb.com,"{\n ""subject_hash_info"": {\n ""hashed_tokens"": [\n ""p5tq83b6R1Hhl0Gy9cN1""\n ],\n ""...",__ Island homes you should see,False,1
2,9dbe29ee-b753-3acf-a105-51c10ab51a48,2026-09-04,2036b338-86a8-39ea-b8a5-f680da841838,5f66d638-dbdc-36ff-9605-46d8a47ab76f,NOT_A_RECEIPT,0.999785,I,None,ab9c17e4-b98c-5b70-bbb1-ce98bb970880,Protect your __ stay with __ Damage Coverage,discover@airbnb.com,"{\n ""subject_hash_info"": {\n ""hashed_tokens"": [\n ""U1L0JO0sGz3eb0tmu0Zv"",\n ""L095C...",Protect your __ stay with __ Damage Coverage,False,2
3,2f0bff85-cd68-3869-bc58-4cbbf7df2c27,2026-09-03,72707140-912c-335c-976f-bb60b444de79,cf3e70ef-91a8-39a8-a454-bd966986e85b,NOT_A_RECEIPT,0.999992,I,None,e8cc996d-c82e-53e6-9c09-5aae476b1623,"Same-day inquiry for Stylish, __ Home (The Lakes), Near Red Rock for today through September 4, ...",automated@airbnb.com,"{\n ""subject_hash_info"": {\n ""hashed_tokens"": [\n ""WbQarH1YsY26rSUwWxX5""\n ],\n ""...","Same-day inquiry for Stylish, __ Home (The Lakes), Near Red Rock for today through September #, #",False,1
4,efccacd7-0ed0-30e3-8e8f-56a4a64d4d91,2026-09-02,26757476-1001-31c8-8923-6634e18652d0,f13710b7-462a-3522-add0-f842b80a75e6,SHIPPING,0.148903,I,"{\n ""merchant_details"": {\n ""merchant_id"": ""1501"",\n ""merchant_name"": ""AirBnB""\n }\n}",75fa67c2-b6ee-514e-b84f-9e63fb842749,Reservation reminder: __ is coming soon!,automated@airbnb.com,"{\n ""subject_hash_info"": {\n ""hashed_tokens"": [\n ""2NDnroGWGplQVHtB60sw""\n ],\n ""...",Reservation reminder: __ is coming soon!,False,1


## 2. Template summary

In [4]:
templates = (
    week.groupby("template")
    .agg(emails=("email_id", "size"), mailboxes=("mailbox_id", "nunique"), hotel_kw=("hotel_kw", "sum"),
         senders=("from_addr", lambda s: ", ".join(s.value_counts().index[:3])),
         email_types=("email_type", lambda s: ", ".join(s.dropna().astype(str).value_counts().index[:3])),
         example=("subject", "first"))
    .sort_values("emails", ascending=False)
    .reset_index()
)
templates.to_csv(OUT / "template_summary.csv", index=False)
templates.head(40)

,template,emails,mailboxes,hotel_kw,senders,email_types,example
0,"Reservation reminder - September #, #",13034,11431,0,automated@airbnb.com,NOT_A_RECEIPT,"Reservation reminder - September 5, 2026"
1,"🏡 __ homes, just for you",6711,6665,0,discover@airbnb.com,NOT_A_RECEIPT,"🏡 __ homes, just for you"
2,Write a review for __,6194,5523,0,automated@airbnb.com,NOT_A_RECEIPT,Write a review for __
3,Reservation reminder: __ is coming soon!,6137,2449,0,automated@airbnb.com,"SHIPPING, NOT_A_RECEIPT",Reservation reminder: __ is coming soon!
4,Log in to Airbnb to keep your account,5665,5028,0,automated@airbnb.com,NOT_A_RECEIPT,Log in to Airbnb to keep your account
5,Write a review for __ group,5005,2480,0,automated@airbnb.com,NOT_A_RECEIPT,Write a review for __ group
6,__ wrote you a review,4407,3355,0,automated@airbnb.com,NOT_A_RECEIPT,__ wrote you a review
7,Your receipt from Airbnb,4304,3795,0,automated@airbnb.com,ORDER,Your receipt from Airbnb
8,Found: Stays with standout features 🔎,4179,4106,0,discover@airbnb.com,NOT_A_RECEIPT,Found: Stays with standout features 🔎
9,Thanks for being a great guest!,4065,3837,0,automated@airbnb.com,NOT_A_RECEIPT,Thanks for being a great guest!


In [5]:
# By sender and NIQ's email_type: is there any field beyond the subject that separates stay types?
(week.groupby(["from_addr", "email_type"], dropna=False)
     .agg(emails=("email_id", "size"), hotel_kw=("hotel_kw", "sum"))
     .sort_values("emails", ascending=False).head(25))

emails  hotel_kw
from_addr                                        email_type                     
discover@airbnb.com                              NOT_A_RECEIPT  111702        77
automated@airbnb.com                             NOT_A_RECEIPT  103762       455
                                                 ORDER           23551        33
                                                 PAYMENT         11106        82
                                                 WELCOME          7902         2
                                                 SHIPPING         6446         0
no-reply@supportmessaging.airbnb.com             NOT_A_RECEIPT    3488         0
automated@airbnb.com                             CANCELLATION     3281         0
express@airbnb.com                               NOT_A_RECEIPT    2764         7
discover@airbnb.com                              ORDER            2316         0
automated@airbnb.com                             RETURNS          1106         0
-{-n-,7*%* "7*,-0@airbnb.com                     NOT_A_RECEIPT     761         0
notifications@mail-hostcommunity.airbnb.com      NOT_A_RECEIPT     561         0
reply@email-support.airbnb.com                   NOT_A_RECEIPT     354         0
resolutions@airbnb.com                           NOT_A_RECEIPT     178         0
no-reply@airbnb.com                              NOT_A_RECEIPT     141         0
express@airbnb.com                               ORDER             125         1
                                                 PAYMENT           125         1
discover@airbnb.com                              WELCOME           123         0
pjmdofklpweffkbufzlvbgivpwfg@events.airbnb.com   NOT_A_RECEIPT      64         0
research@research.airbnb.com                     NOT_A_RECEIPT      59         0
+,07*-$.,7+&1+,,'%*-'*-$!"/"- @events.airbnb.com NOT_A_RECEIPT      39         0
4&07n ,"07n!&" +@airbnb.com                      NOT_A_RECEIPT      37         0
discover@airbnb.com                              CANCELLATION       37         0
"*1!-!n%1*&-'/:@airbnb.com                       NOT_A_RECEIPT      29         0

In [6]:
# What do the two aux_data columns hold? Distinct top-level key sets of each.
def keys(col: str) -> pd.Series:
    parsed = week[col].dropna().map(lambda v: json.loads(v) if isinstance(v, str) else v)
    return parsed.map(lambda d: tuple(sorted(d)) if isinstance(d, dict) else type(d).__name__).value_counts()


print("email_aux:\n", keys("email_aux").head(10), "\n")
print("header_aux:\n", keys("header_aux").head(10))

email_aux:
 email_aux
(merchant_details, order_details)    24771
(merchant_details,)                  14502
(order_details,)                      4196
Name: count, dtype: int64 



header_aux:
 header_aux
(subject_hash_info,)    280640
Name: count, dtype: int64


## 3. Stratified sample + every hotel-keyword email

In [7]:
sample = (week.groupby("template", group_keys=False)
              .apply(lambda g: g.sample(min(len(g), SAMPLE_PER_TEMPLATE), random_state=SEED)))
hotel = week[week.hotel_kw]

EXPORT_COLS = ["email_sent_date", "mailbox_id", "user_id", "from_addr", "email_type", "email_type_confidence",
               "email_status", "template", "subject", "hotel_kw", "n_blanks", "subject_id", "email_id",
               "header_aux", "email_aux"]
sample.sort_values(["template", "email_sent_date"])[EXPORT_COLS].to_csv(OUT / "sample_by_template.csv", index=False)
hotel.sort_values(["template", "email_sent_date"])[EXPORT_COLS].to_csv(OUT / "hotel_keyword_emails.csv", index=False)
print(f"sample: {len(sample):,} rows over {sample.template.nunique():,} templates; hotel-keyword: {len(hotel):,} rows")
hotel.template.value_counts().head(30)

C:\Users\fintan.creedon\AppData\Local\Temp\ipykernel_42464\2280705320.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(len(g), SAMPLE_PER_TEMPLATE), random_state=SEED)))


sample: 70,129 rows over 28,921 templates; hotel-keyword: 658 rows


template
__ un # % de __ con tu hotel                                                                     37
Get #% credit on hotel bookings                                                                  26
Write a review for __ Hotel                                                                       9
Pending: Reservation Request at The Bus Stop Inn ## for Sep # – #, #                              5
Write __ __ Hotel __ review by September #                                                        4
Pending: Reservation Request at The Bus Stop Inn ## __ Bed Private Apartment for Oct # – #, #     4
Final reminder: Write a review for The __ Hotel                                                   4
Write a review for Hotel                                                                          4
__ Pedido de Reserva em __ resort __ à __ __ - __ para # de __ de # – # de jan. de #              4
Pending: Reservation Request at The Bus Stop Inn ## for Oct # – #, #                       

## 4. Mailbox traces

Mailboxes that got a first review request ("Write a review for …" / "Write … review by …") this week: `N_TRACE_MAILBOXES` with a hotel-like host name, and the same number with a non-hotel one. Pull every Airbnb email each got over the prior `TRACE_LOOKBACK_DAYS` to see what the booking, reminder and stay emails looked like.

In [8]:
is_review = week.subject.str.match(r"(?i)write (a review for |.+ review by )")
reviews = week[is_review]
hotel_mb = reviews[reviews.hotel_kw].mailbox_id.drop_duplicates()
other_mb = reviews[~reviews.hotel_kw & ~reviews.mailbox_id.isin(hotel_mb)].mailbox_id.drop_duplicates()

picks = pd.concat([
    pd.DataFrame({"mailbox_id": hotel_mb.sample(min(len(hotel_mb), N_TRACE_MAILBOXES), random_state=SEED), "group": "hotel_kw"}),
    pd.DataFrame({"mailbox_id": other_mb.sample(min(len(other_mb), N_TRACE_MAILBOXES), random_state=SEED), "group": "other"}),
])
mb_list = ", ".join(f"'{m}'" for m in picks.mailbox_id)

traces = q(f"""
    WITH e AS (SELECT * FROM {EMAILS}
               WHERE email_sent_date BETWEEN DATEADD(day, -{TRACE_LOOKBACK_DAYS}, '{WEEK_START}'::DATE) AND '{WEEK_END}'
                 AND mailbox_id IN ({mb_list})),
         h AS (SELECT subject_id, subject, from_addr, aux_data FROM {HEADER} WHERE domain = 'airbnb')
    SELECT {COLS}
    FROM e JOIN h ON e.subject_id = h.subject_id
""")
traces = enrich(traces).merge(picks, on="mailbox_id").sort_values(["group", "mailbox_id", "email_sent_date"])
traces[["group"] + EXPORT_COLS].to_csv(OUT / "mailbox_traces.csv", index=False)
print(f"{len(traces):,} emails across {traces.mailbox_id.nunique()} mailboxes")
traces.groupby("group").template.value_counts().groupby(level=0).head(15)

2,541 emails across 50 mailboxes


C:\Users\fintan.creedon\AppData\Local\Temp\ipykernel_42464\806143361.py:19: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["hotel_kw"] = df["subject"].str.contains(HOTEL_RE, case=False, regex=True)


group     template                                                        
hotel_kw  Reservation reminder - September #, #                                39
          New message from Airbnb Support                                      31
          Confirmed: Your September # – # trip, here’s your Airbnb receipt     25
          Welcome to __ __                                                     25
          Your receipt from Airbnb                                             23
          Reservation reminder - August #, #                                   19
          Thanks for being a great guest!                                      15
          Write a review for __                                                15
          🏡 __ homes, just for you                                             15
          __ homes you should see                                              13
          Confirmed: Your August # – # trip, here’s your Airbnb receipt        10
          Your Airbnb c

In [9]:
# Read one hotel-group mailbox end to end.
mb = picks[picks.group == "hotel_kw"].mailbox_id.iloc[0]
traces[traces.mailbox_id == mb][["email_sent_date", "from_addr", "email_type", "subject"]]

,email_sent_date,from_addr,email_type,subject
2028,2026-05-10,discover@airbnb.com,NOT_A_RECEIPT,__ homes you should see
2043,2026-05-14,discover@airbnb.com,NOT_A_RECEIPT,__ homes you should see
2024,2026-07-08,discover@airbnb.com,NOT_A_RECEIPT,Lake __ stays you should see
1718,2026-07-19,discover@airbnb.com,NOT_A_RECEIPT,El __ stays you should see
2435,2026-07-20,discover@airbnb.com,NOT_A_RECEIPT,"🏡 El __ homes, just for you"
96,2026-07-24,discover@airbnb.com,NOT_A_RECEIPT,Thank the hosts for discounted homes
1183,2026-07-25,discover@airbnb.com,NOT_A_RECEIPT,"🏡 __ __ del Sol homes, just for you"
539,2026-07-30,discover@airbnb.com,NOT_A_RECEIPT,__ __ del Sol homes you should see
1947,2026-08-02,automated@airbnb.com,ORDER,Confirmed: Your reservation for Sep 5 – 7
393,2026-08-03,discover@airbnb.com,ORDER,Protect your trip with travel insurance


In [10]:
sorted(p.name for p in OUT.iterdir())

['hotel_keyword_emails.csv',
 'mailbox_traces.csv',
 'sample_by_template.csv',
 'template_summary.csv']